In [55]:
import os
import sys
import json
import glob
import gc

import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType

snowflake_conn_prop = Snowflake_configuration.ds1_role_json
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

In [83]:
import warnings 
warnings.filterwarnings('ignore')

In [50]:
#Checking SKU_SUPERCEDENCE table
sku_supercedence_history = session.sql('SELECT MAX(UPDATED_ON) FROM MOP_DATABASE.SOQ.SKU_SUPERCEDENCE_HISTORY')

In [51]:
# def fetchSKUSupercedence_snowpark(session,SKU_SUPERCEDENCE_MODEL_FAMILY):
def fetchSKUSupercedence_snowpark(session):
    data = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")
    data_1 = session.table("MOP_DATABASE.SOQ.MODEL_FAMILY_MAPPING") 
    result = data.join(data_1, on="MODEL", how="left") 
    result = result.with_column("SKU_UNIQUE_FAMILY_CODE", F.col("UNIQUEFAMILYCODE")) 

    for old_col in result.columns:
        new_col = old_col.replace('"','')
        result = result.rename(old_col, new_col) 
    
    result = result.with_column("MODEL_FAMILY_CODE",
                            F.concat(F.col("MODEL_FAMILY"), F.lit('<>'),
                            F.substring(F.col("UNIQUEFAMILYCODE"),
                                F.charindex(F.lit('<>'), F.col("UNIQUEFAMILYCODE")) + F.lit(2))
                                )) 
    
    result = result.rename("UNIQUEFAMILYCODE", "UNIQUE FAMILY CODE") 
    # result.write.mode("overwrite").save_as_table(SKU_SUPERCEDENCE_MODEL_FAMILY) 
    return result

result = fetchSKUSupercedence_snowpark(session)


In [52]:
def return_models_for_forecasting(session, use_selected_models, logger):
    """
    Returns a list of model names or None based on user filter parameter settings.
    """
    if not use_selected_models:
        # logger.info("Model filter disabled — considering ALL models in ECR sales.")
        return None

    models_for_forecasting = session.table('MOP_DATABASE.SOQ.MODELS_FOR_FORECASTING').to_pandas()
    name_of_models = models_for_forecasting["MODEL_NAME"].tolist()
    # logger.info("Model filter enabled — %s models selected for forecasting.", len(name_of_models))
    return name_of_models

name_of_models = return_models_for_forecasting(session,True,None)

In [53]:
def get_ecr_sales_snowpark(session, customer_types, start_date, name_of_models,end_date):
    ecr_sales = session.table("ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS") \
        .filter(F.col("X_CUSTOMER_TYPE").in_(customer_types)) \
        .filter((F.col("CAL_DATE") >= F.lit(start_date)) & (F.col("CAL_DATE") <= F.lit(end_date)))
        
    if name_of_models is not None:
        ecr_sales = ecr_sales.filter(F.col("MODEL").isin(name_of_models))
        
    ecr_sales = ecr_sales.with_column("NET_SALES", 
    F.when(
        (F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")) < 0, 
        F.lit(0)
    ).otherwise(
        F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")
    )
)


    return ecr_sales

### Historical sales

In [29]:
ecr_sales = get_ecr_sales_snowpark(session,start_date='2026-06-01',end_date='2026-08-31',customer_types=['Individual'],name_of_models=name_of_models)
ecr_sales = ecr_sales.select('DEALER_CODE','CAL_DATE','SKU','NET_SALES')
active_sku_supercedence = result.filter(F.lower(F.col("SKUSTATUS")) == 'active')
active_sku_supercedence = active_sku_supercedence.select("SKU","MODEL_FAMILY_CODE")
obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW") 
sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")

obd_data_joined = obd_data.join(
    sku_supercedence.select("SKU", "SKUSTATUS"), 
    obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"], 
    how='left'
)

obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
                                        .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU") 


ecr_sales = ecr_sales.join(obd_data_active_skus, ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"], how="left")
ecr_sales = ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))

joined_df = ecr_sales.join(right=active_sku_supercedence,on=["SKU"])

joined_df = joined_df.select("DEALER_CODE","CAL_DATE","MODEL_FAMILY_CODE","SKU","NET_SALES")


In [37]:

dealer_master = session.sql("""SELECT TRIM(PARENT_DEALER_CODE) AS PARENT_DEALER_CODE,TRIM(DEALER_CODE) AS DEALER_CODE
                            FROM ANALYTICS_DATABASE.ANALYTICS_SALES.VW_DEALER_MASTER
                            WHERE NOT REGEXP_LIKE(DEALER_CODE,'^17.*')
                            AND LOWER(ORG_STATUS) = 'active'
                            AND DEALER_DFNC = 'Y'
                            """)

family_level_daily_sales = dealer_master.join(joined_df,on=["DEALER_CODE"])

family_level_daily_sales = family_level_daily_sales.with_column("PARENT_DEALER_CODE_MODEL_FAMILY",F.concat(F.trim(F.col("PARENT_DEALER_CODE")),F.lit('<>'),F.col("MODEL_FAMILY_CODE")))


family_sales_3_months = family_level_daily_sales.group_by("PARENT_DEALER_CODE_MODEL_FAMILY").agg(F.sum("NET_SALES").alias("LAST_3_MONTHS_FAMILY_SALES"))

sku_level_sales_3_months = family_level_daily_sales.group_by("PARENT_DEALER_CODE_MODEL_FAMILY","SKU").agg(F.sum("NET_SALES").alias("LAST_3_MONTHS_SKU_SALES"))

family_and_sku_with_last_3_months_sales = family_sales_3_months.join(right=sku_level_sales_3_months,on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

sku_count = family_and_sku_with_last_3_months_sales.group_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .agg(F.count("SKU").alias("SKU_COUNT"))

family_and_sku_with_last_3_months_sales = family_and_sku_with_last_3_months_sales.join(sku_count, on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

family_and_sku_with_last_3_months_sales = family_and_sku_with_last_3_months_sales.with_column(
    "PROPORTION",
    F.when(F.col("LAST_3_MONTHS_FAMILY_SALES") != 0,
           F.col("LAST_3_MONTHS_SKU_SALES") / F.col("LAST_3_MONTHS_FAMILY_SALES"))
    .otherwise(F.lit(1) / F.col("SKU_COUNT"))
)

In [ ]:
dealer_master = dealer_master.select(F.trim("PARENT_DEALER_CODE").alias("PARENT_DEALER_CODE"),F.trim(F.col("DEALER_CODE")).alias("DEALER_CODE"))


family_and_sku_with_last_3_months_sales = family_and_sku_with_last_3_months_sales.with_column("PARENT_DEALER_CODE",F.trim(F.split_part(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),F.lit('<>'),F.lit(1))))
family_and_sku_with_last_3_months_sales_joined = family_and_sku_with_last_3_months_sales.join(dealer_master,on=["PARENT_DEALER_CODE"])

dealer_sku_sales = family_and_sku_with_last_3_months_sales_joined.select(F.col("DEALER_CODE"),F.col("SKU"),F.col("LAST_3_MONTHS_SKU_SALES"))


from snowflake.snowpark import Window

# Rank SKUs within each dealer by sales descending
w = Window.partition_by("DEALER_CODE").order_by(F.col("LAST_3_MONTHS_SKU_SALES").desc())

# Total sales per dealer
dealer_total = Window.partition_by("DEALER_CODE")

dealer_sku_sales = dealer_sku_sales.with_column(
    "CUMULATIVE_PCT",
    F.sum("LAST_3_MONTHS_SKU_SALES").over(w.rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW))
    / F.sum("LAST_3_MONTHS_SKU_SALES").over(dealer_total)
)

dealer_sku_sales = dealer_sku_sales.with_column(
    "ABC_CATEGORY",
    F.when(F.col("CUMULATIVE_PCT") <= 0.80, F.lit("A"))
     .when(F.col("CUMULATIVE_PCT") <= 0.95, F.lit("B"))
     .otherwise(F.lit("C"))
)

family_and_sku_with_last_3_months_sales_joined_final = family_and_sku_with_last_3_months_sales_joined.join(dealer_sku_sales.select("DEALER_CODE","SKU","ABC_CATEGORY"),on=["DEALER_CODE","SKU"])

final_historical_sales_df = family_and_sku_with_last_3_months_sales_joined_final.select('PARENT_DEALER_CODE_MODEL_FAMILY','SKU','PROPORTION','ABC_CATEGORY','LAST_3_MONTHS_FAMILY_SALES','LAST_3_MONTHS_SKU_SALES')

### Actual sales

In [33]:
actual_ecr_sales = get_ecr_sales_snowpark(session,start_date='2026-09-01',end_date='2026-09-22',customer_types=['Individual'],name_of_models=name_of_models)

actual_ecr_sales = actual_ecr_sales.select('DEALER_CODE','CAL_DATE','SKU','NET_SALES')

obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW") 
sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")

obd_data_joined = obd_data.join(
    sku_supercedence.select("SKU", "SKUSTATUS"), 
    obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"], 
    how='left'
)

obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
                                        .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU") 


actual_ecr_sales = actual_ecr_sales.join(obd_data_active_skus, actual_ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"], how="left")
actual_ecr_sales = actual_ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))





In [34]:
#OBD_mapping
obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW") 
sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")

obd_data_joined = obd_data.join(
    sku_supercedence.select("SKU", "SKUSTATUS"), 
    obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"], 
    how='left'
)

obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
                                        .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU") 

actual_joined_df = actual_ecr_sales.join(right=active_sku_supercedence,on=["SKU"])

actual_joined_df = actual_joined_df.select("DEALER_CODE","CAL_DATE","MODEL_FAMILY_CODE","SKU","NET_SALES")

actual_joined_df_with_pdc = dealer_master.join(actual_joined_df,on=["DEALER_CODE"])
actual_joined_df_with_pdc = actual_joined_df_with_pdc.drop("DEALER_CODE")
actual_joined_df_with_pdc = actual_joined_df_with_pdc.with_column("PARENT_DEALER_CODE_MODEL_FAMILY",F.concat(F.col("PARENT_DEALER_CODE"),F.lit('<>'),F.col("MODEL_FAMILY_CODE")))



In [35]:
actual_sales_df = actual_joined_df_with_pdc.select(F.col('PARENT_DEALER_CODE_MODEL_FAMILY'),F.col("SKU"),F.col("CAL_DATE"),F.col("NET_SALES"))
actual_sales_df.show()

-----------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"                   |"SKU"           |"CAL_DATE"           |"NET_SALES"  |
-----------------------------------------------------------------------------------------------------------
|10382<>SPLENDOR+<>DRUM<>SELF<>CAST<>RED BLACK       |HSPLMDRSCFISBK  |2026-09-01 00:00:00  |1.000000     |
|10030<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK RED P...  |HSPLMDRSCFIRPB  |2026-09-08 00:00:00  |1.000000     |
|12276<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK RED P...  |HSPLMDRSCFIRPB  |2026-09-13 00:00:00  |1.000000     |
|12276<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK RED P...  |HSPLMDRSCFIRPB  |2026-09-20 00:00:00  |1.000000     |
|11410<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE            |HSPLMTRSCFIBSB  |2026-09-11 00:00:00  |1.000000     |
|12276<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK           |HSPLMDRSCFIBHG  |2026-09-05 00:00:00  |1.000000     |
|12254<>SPLENDOR+<>DRUM<>SEL

In [36]:
final_historical_sales_df.show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"              |"SKU"           |"PROPORTION"    |"ABC_CATEGORY"  |"LAST_3_MONTHS_FAMILY_SALES"  |"LAST_3_MONTHS_SKU_SALES"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------
|10382<>XOOM<>DRUM<>SELF<>CAST<>BLUE            |HXOMFDSSCFIVBU  |1.000000000000  |C               |1.000000                      |1.000000                   |
|10382<>XOOM<>DRUM<>SELF<>CAST<>BLUE            |HXOMFDSSCFIVBU  |1.000000000000  |B               |1.000000                      |1.000000                   |
|10382<>XOOM<>DRUM<>SELF<>CAST<>BLUE            |HXOMFDSSCFIVBU  |1.000000000000  |C               |1.000000                      |1.000000                   |
|10382<>XOOM<>DRUM<>SELF<>CAST<>BLUE    

In [58]:
#Join with prediction
pred_itr_3 = pd.read_parquet(r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#3_feature_engineering\Modelling\predictions_2026_negbin_torch_convention\20260922_153432_34f1cdf5\DA7CA8~1.PAR")
pred_itr_3_mean = pred_itr_3[["PARENT_DEALER_CODE_MODEL_FAMILY","CAL_DATE","PRED_MEAN"]]
pred_itr_3_sf = session.create_dataframe(pred_itr_3_mean)

In [40]:
pred_itr_3_sf.show()

------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"   |"CAL_DATE"           |"PRED_MEAN"            |
------------------------------------------------------------------------------------
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-01 00:00:00  |0.0017402732428584413  |
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-02 00:00:00  |0.0010299685642882134  |
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-03 00:00:00  |0.0012483010995808482  |
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-04 00:00:00  |0.04884695651224624    |
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-05 00:00:00  |0.006636736064024455   |
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-06 00:00:00  |0.0006229028339180325  |
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-07 00:00:00  |0.008748276691695372   |
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-08 00:00:00  |0.010672717265009906   |
|10915_DESTINI_DISC_SELF_CAST_BLACK  |2026-09-09 00:00:00  |0.013

In [59]:
pred_itr_3_sf = pred_itr_3_sf.with_column("PARENT_DEALER_CODE_MODEL_FAMILY",F.replace(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),F.lit('_'),F.lit('<>')))
pred_itr_3_sf.show()

-----------------------------------------------------------------------------------------
|"CAL_DATE"           |"PRED_MEAN"            |"PARENT_DEALER_CODE_MODEL_FAMILY"        |
-----------------------------------------------------------------------------------------
|2026-09-01 00:00:00  |0.0017402732428584413  |10915<>DESTINI<>DISC<>SELF<>CAST<>BLACK  |
|2026-09-02 00:00:00  |0.0010299685642882134  |10915<>DESTINI<>DISC<>SELF<>CAST<>BLACK  |
|2026-09-03 00:00:00  |0.0012483010995808482  |10915<>DESTINI<>DISC<>SELF<>CAST<>BLACK  |
|2026-09-04 00:00:00  |0.04884695651224624    |10915<>DESTINI<>DISC<>SELF<>CAST<>BLACK  |
|2026-09-05 00:00:00  |0.006636736064024455   |10915<>DESTINI<>DISC<>SELF<>CAST<>BLACK  |
|2026-09-06 00:00:00  |0.0006229028339180325  |10915<>DESTINI<>DISC<>SELF<>CAST<>BLACK  |
|2026-09-07 00:00:00  |0.008748276691695372   |10915<>DESTINI<>DISC<>SELF<>CAST<>BLACK  |
|2026-09-08 00:00:00  |0.010672717265009906   |10915<>DESTINI<>DISC<>SELF<>CAST<>BLACK  |
|2026-09-0

In [60]:
join_with_predictions = final_output.join(pred_itr_3_sf,on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

In [61]:
# final_output = final_output.with_column("LENGTH",F.length("PARENT_DEALER_CODE_MODEL_FAMILY"))


In [43]:
join_with_predictions.show()

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"               |"SKU"           |"PROPORTION"    |"ABC_CATEGORY"  |"LAST_3_MONTHS_FAMILY_SALES"  |"LAST_3_MONTHS_SKU_SALES"  |"CAL_DATE"           |"PRED_MEAN"          |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|10918<>SUPER SPLENDOR<>DRUM<>SELF<>CAST<>BLACK  |HSPSEDRSCFIGBL  |1.000000000000  |A               |2.000000                      |2.000000                   |2026-09-01 00:00:00  |0.08998872777666858  |
|10918<>SUPER SPLENDOR<>DRUM<>SELF<>CAST<>BLACK  |HSPSEDRSCFIGBL  |1.000000000000  |A               |2.000000                      |2.000000                   |2026-09-02 00:00:00 

In [44]:
list_of_columns = ['PARENT_DEALER_CODE_MODEL_FAMILY','SKU','CAL_DATE','PRED_MEAN','PROPORTION','ABC_CATEGORY']
join_with_predictions_sf = join_with_predictions.select('PARENT_DEALER_CODE_MODEL_FAMILY','SKU','CAL_DATE','PRED_MEAN','PROPORTION','ABC_CATEGORY')

join_with_predictions_sf = join_with_predictions_sf.with_column("SKU_PREDICTION",F.col("PROPORTION")*F.col("PRED_MEAN"))

join_with_predictions_sf.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"          |"SKU"           |"CAL_DATE"           |"PRED_MEAN"         |"PROPORTION"    |"ABC_CATEGORY"  |"SKU_PREDICTION"    |
----------------------------------------------------------------------------------------------------------------------------------------------------------------
|12254<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK  |HSPLMDRSCFIBHG  |2026-09-01 00:00:00  |2.988904369246057   |1.000000000000  |A               |2.988904369246057   |
|12254<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK  |HSPLMDRSCFIBHG  |2026-09-02 00:00:00  |4.0929890213636755  |1.000000000000  |A               |4.0929890213636755  |
|12254<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK  |HSPLMDRSCFIBHG  |2026-09-03 00:00:00  |4.684421202460296   |1.000000000000  |A               |4.684421202460296   |
|12254<>SPLENDOR+<>DRUM<>SELF<>CAS

In [ ]:
join_with_predictions_sf.select(F.sum("SKU_PREDICTION").alias("TOTAL_SALES_AT_SKU_LEVEL"),F.sum("PRED_MEAN").alias("TOTAL_SALES_AT_FAMILY_LEVEL")).show()

--------------------------------------------------------------
|"TOTAL_SALES_AT_SKU_LEVEL"  |"TOTAL_SALES_AT_FAMILY_LEVEL"  |
--------------------------------------------------------------
|137242.9642379012           |212765.96628734874             |
--------------------------------------------------------------



In [47]:
# This is the correct total
join_with_predictions_sf.select(
    F.sum("SKU_PREDICTION").alias("TOTAL_SKU_PREDICTION")
).show()

# To get the true family-level total for comparison, deduplicate first
pred_itr_3_sf.select(
    F.sum("PRED_MEAN").alias("TOTAL_FAMILY_PREDICTION")
).show()

--------------------------
|"TOTAL_SKU_PREDICTION"  |
--------------------------
|137242.9642379012       |
--------------------------

-----------------------------
|"TOTAL_FAMILY_PREDICTION"  |
-----------------------------
|2073623.4918465866         |
-----------------------------



### Edited code

In [56]:
# ============================================================
# 1. SKU Supercedence + Model Family Mapping
# ============================================================
def fetchSKUSupercedence_snowpark(session):
    data = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")
    data_1 = session.table("MOP_DATABASE.SOQ.MODEL_FAMILY_MAPPING")
    result = data.join(data_1, on="MODEL", how="left")
    result = result.with_column("SKU_UNIQUE_FAMILY_CODE", F.col("UNIQUEFAMILYCODE"))
    for old_col in result.columns:
        new_col = old_col.replace('"', '')
        result = result.rename(old_col, new_col)
    result = result.with_column(
        "MODEL_FAMILY_CODE",
        F.concat(
            F.col("MODEL_FAMILY"),
            F.lit('<>'),
            F.substring(
                F.col("UNIQUEFAMILYCODE"),
                F.charindex(F.lit('<>'), F.col("UNIQUEFAMILYCODE")) + F.lit(2)
            )
        )
    )
    result = result.rename("UNIQUEFAMILYCODE", "UNIQUE FAMILY CODE")
    return result

result = fetchSKUSupercedence_snowpark(session)

# ============================================================
# 2. Models for Forecasting
# ============================================================
def return_models_for_forecasting(session, use_selected_models):
    if not use_selected_models:
        return None
    models_for_forecasting = session.table('MOP_DATABASE.SOQ.MODELS_FOR_FORECASTING').to_pandas()
    return models_for_forecasting["MODEL_NAME"].tolist()

name_of_models = return_models_for_forecasting(session, True)

# ============================================================
# 3. ECR Sales
# ============================================================
def get_ecr_sales_snowpark(session, customer_types, start_date, end_date, name_of_models):
    ecr_sales = session.table("ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS") \
        .filter(F.col("X_CUSTOMER_TYPE").in_(customer_types)) \
        .filter((F.col("CAL_DATE") >= F.lit(start_date)) & (F.col("CAL_DATE") <= F.lit(end_date)))
    if name_of_models is not None:
        ecr_sales = ecr_sales.filter(F.col("MODEL").isin(name_of_models))
    ecr_sales = ecr_sales.with_column(
        "NET_SALES",
        F.when(
            (F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")) < 0,
            F.lit(0)
        ).otherwise(
            F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")
        )
    )
    return ecr_sales

ecr_sales = get_ecr_sales_snowpark(
    session,
    start_date='2026-06-01',
    end_date='2026-08-31',
    customer_types=['Individual'],
    name_of_models=name_of_models
)
ecr_sales = ecr_sales.select('DEALER_CODE', 'CAL_DATE', 'SKU', 'NET_SALES')

# ============================================================
# 4. Active SKU Supercedence
# ============================================================
active_sku_supercedence = result.filter(F.lower(F.col("SKUSTATUS")) == 'active')
active_sku_supercedence = active_sku_supercedence.select("SKU", "MODEL_FAMILY_CODE")

# ============================================================
# 5. OBD Mapping - Map old SKUs to current active SKUs
# ============================================================
obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW")
sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")

obd_data_joined = obd_data.join(
    sku_supercedence.select("SKU", "SKUSTATUS"),
    obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"],
    how='left'
)
obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
    .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU")

ecr_sales = ecr_sales.join(
    obd_data_active_skus,
    ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"],
    how="left"
)
ecr_sales = ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))

# ============================================================
# 6. Join sales with active SKU supercedence (inner join)
# ============================================================
joined_df = ecr_sales.join(right=active_sku_supercedence, on=["SKU"])
joined_df = joined_df.select("DEALER_CODE", "CAL_DATE", "MODEL_FAMILY_CODE", "SKU", "NET_SALES")

# ============================================================
# 7. Dealer Master
# ============================================================
dealer_master = session.sql("""
    SELECT TRIM(PARENT_DEALER_CODE) AS PARENT_DEALER_CODE,
           TRIM(DEALER_CODE) AS DEALER_CODE
    FROM ANALYTICS_DATABASE.ANALYTICS_SALES.VW_DEALER_MASTER
    WHERE NOT REGEXP_LIKE(DEALER_CODE, '^17.*')
      AND LOWER(ORG_STATUS) = 'active'
      AND DEALER_DFNC = 'Y'
""")

# ============================================================
# 8. Family-level daily sales (dealer grain)
# ============================================================
family_level_daily_sales = dealer_master.join(joined_df, on=["DEALER_CODE"])
family_level_daily_sales = family_level_daily_sales.with_column(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    F.concat(F.trim(F.col("PARENT_DEALER_CODE")), F.lit('<>'), F.col("MODEL_FAMILY_CODE"))
)

# ============================================================
# 9. Aggregate at parent_dealer+family and parent_dealer+family+SKU
# ============================================================
family_sales_3_months = family_level_daily_sales \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .agg(F.sum("NET_SALES").alias("LAST_3_MONTHS_FAMILY_SALES"))

sku_level_sales_3_months = family_level_daily_sales \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY", "SKU") \
    .agg(F.sum("NET_SALES").alias("LAST_3_MONTHS_SKU_SALES"))

family_and_sku = family_sales_3_months.join(
    right=sku_level_sales_3_months,
    on=["PARENT_DEALER_CODE_MODEL_FAMILY"]
)

# ============================================================
# 10. SKU Proportion within each family (with equal-split fallback)
# ============================================================
sku_count = family_and_sku \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .agg(F.count("SKU").alias("SKU_COUNT"))

family_and_sku = family_and_sku.join(sku_count, on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

family_and_sku = family_and_sku.with_column(
    "PROPORTION",
    F.when(F.col("LAST_3_MONTHS_FAMILY_SALES") != 0,
           F.col("LAST_3_MONTHS_SKU_SALES") / F.col("LAST_3_MONTHS_FAMILY_SALES"))
    .otherwise(F.lit(1) / F.col("SKU_COUNT"))
)

# ============================================================
# 11. ABC Classification at dealer+SKU level (from actual dealer sales)
# ============================================================
from snowflake.snowpark import Window

dealer_sku_sales = family_level_daily_sales \
    .group_by("DEALER_CODE", "SKU") \
    .agg(F.sum("NET_SALES").alias("DEALER_SKU_SALES"))

w = Window.partition_by("DEALER_CODE").order_by(F.col("DEALER_SKU_SALES").desc())
dealer_total = Window.partition_by("DEALER_CODE")

dealer_sku_sales = dealer_sku_sales.with_column(
    "CUMULATIVE_PCT",
    F.sum("DEALER_SKU_SALES").over(w.rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW))
    / F.sum("DEALER_SKU_SALES").over(dealer_total)
)

dealer_sku_sales = dealer_sku_sales.with_column(
    "ABC_CATEGORY",
    F.when(F.col("CUMULATIVE_PCT") <= 0.80, F.lit("A"))
     .when(F.col("CUMULATIVE_PCT") <= 0.95, F.lit("B"))
     .otherwise(F.lit("C"))
).select("DEALER_CODE", "SKU", "ABC_CATEGORY")

# ============================================================
# 12. Join ABC back to parent-level proportions via dealer_master
# ============================================================
proportions_with_dealer = family_and_sku \
    .with_column(
        "PARENT_DEALER_CODE",
        F.trim(F.split_part(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"), F.lit('<>'), F.lit(1)))
    ) \
    .join(dealer_master, on=["PARENT_DEALER_CODE"])

final = proportions_with_dealer.join(dealer_sku_sales, on=["DEALER_CODE", "SKU"])

# ============================================================
# 13. Final output at PARENT_DEALER_CODE_MODEL_FAMILY + SKU level
# ============================================================
final_output = final.select(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    "SKU",
    "PROPORTION",
    "ABC_CATEGORY",
    "LAST_3_MONTHS_FAMILY_SALES",
    "LAST_3_MONTHS_SKU_SALES"
)

In [94]:
# Actual ECR Sales (Sep 1-22)
actual_ecr_sales = get_ecr_sales_snowpark(
    session,
    start_date='2026-09-01',
    end_date='2026-09-22',
    customer_types=['Individual'],
    name_of_models=name_of_models
)
actual_ecr_sales = actual_ecr_sales.select('DEALER_CODE', 'CAL_DATE', 'SKU', 'NET_SALES')

# OBD Mapping (once)
obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW")
sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")

obd_data_joined = obd_data.join(
    sku_supercedence.select("SKU", "SKUSTATUS"),
    obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"],
    how='left'
)
obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
    .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU")

actual_ecr_sales = actual_ecr_sales.join(
    obd_data_active_skus,
    actual_ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"],
    how="left"
)
actual_ecr_sales = actual_ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))

# Join with active SKU supercedence
actual_joined_df = actual_ecr_sales.join(right=active_sku_supercedence, on=["SKU"])
actual_joined_df = actual_joined_df.select("DEALER_CODE", "CAL_DATE", "MODEL_FAMILY_CODE", "SKU", "NET_SALES")

# Join with dealer master to get parent dealer code
actual_joined_df_with_pdc = dealer_master.join(actual_joined_df, on=["DEALER_CODE"])
actual_joined_df_with_pdc = actual_joined_df_with_pdc.with_column(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    F.concat(F.trim(F.col("PARENT_DEALER_CODE")), F.lit('<>'), F.trim(F.col("MODEL_FAMILY_CODE")))
)

# Final actual sales
actual_sales_df = actual_joined_df_with_pdc.select(
    "PARENT_DEALER_CODE_MODEL_FAMILY", "SKU", "CAL_DATE", "NET_SALES"
)
actual_sales_df.show()

-----------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"                   |"SKU"           |"CAL_DATE"           |"NET_SALES"  |
-----------------------------------------------------------------------------------------------------------
|10054<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK RED P...  |HSPLMDRSCFIRPB  |2026-09-10 00:00:00  |1.000000     |
|10054<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK RED P...  |HSPLMDRSCFIRPB  |2026-09-18 00:00:00  |1.000000     |
|11028<>XTREME 125<>DRUM<>SELF<>CAST<>GREY           |HXTRSASSCFIBGY  |2026-09-11 00:00:00  |1.000000     |
|10030<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK RED P...  |HSPLMDRSCFIRPB  |2026-09-14 00:00:00  |1.000000     |
|10592<>SPLENDOR+<>DRUM<>SELF<>CAST<>SILVER          |HSPLMDRSCFIFSV  |2026-09-15 00:00:00  |1.000000     |
|10030<>SPLENDOR+<>DRUM<>SELF<>CAST<>RED             |HSPPLHRSCFIMGM  |2026-09-14 00:00:00  |1.000000     |
|10382<>SPLENDOR+<>DRUM<>SEL

### Predictions

In [84]:
pred_itr_3 = pd.read_parquet(r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#3_feature_engineering\Modelling\predictions_2026_negbin_torch_convention\20260922_153432_34f1cdf5\DA7CA8~1.PAR")
pred_itr_3_mean = pred_itr_3[["PARENT_DEALER_CODE_MODEL_FAMILY","CAL_DATE","PRED_MEAN"]]
pred_itr_3_mean = pred_itr_3_mean.loc[pred_itr_3_mean["CAL_DATE"]<='2026-09-22',:]
pred_itr_3_sf = session.create_dataframe(pred_itr_3_mean)

In [85]:
pred_itr_3_sf = pred_itr_3_sf.with_column("PARENT_DEALER_CODE_MODEL_FAMILY",F.replace(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),F.lit('_'),F.lit('<>')))

In [86]:
join_with_predictions = final_output.join(pred_itr_3_sf,on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

In [88]:
#Series in prediction output but not in historical sales

missing_series = final_output.join(
    pred_itr_3_sf,
    final_output["PARENT_DEALER_CODE_MODEL_FAMILY"] == pred_itr_3_sf["PARENT_DEALER_CODE_MODEL_FAMILY"],
    "leftanti"
)

missing_series.select(F.count_distinct("PARENT_DEALER_CODE_MODEL_FAMILY").alias("SERIES_IN_HISTORICAL_OUTPUT_BUT_NOT_IN_PREDICTIONS")).show()

------------------------------------------------------
|"SERIES_IN_HISTORICAL_OUTPUT_BUT_NOT_IN_PREDICT...  |
------------------------------------------------------
|301                                                 |
------------------------------------------------------



In [89]:
missing_series.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"               |"SKU"           |"PROPORTION"    |"ABC_CATEGORY"  |"LAST_3_MONTHS_FAMILY_SALES"  |"LAST_3_MONTHS_SKU_SALES"  |
----------------------------------------------------------------------------------------------------------------------------------------------------------------
|10030<>DESTINI<>DRUM<>SELF<>SHEET METAL<>WHITE  |HDESHDRLMFIPSW  |1.000000000000  |C               |2.000000                      |2.000000                   |
|10030<>DESTINI<>DRUM<>SELF<>CAST<>WHITE         |HDSTMDRVCFIFWT  |1.000000000000  |C               |2.000000                      |2.000000                   |
|10675<>DESTINI<>DRUM<>SELF<>CAST<>WHITE         |HDSTMDRVCFIFWT  |1.000000000000  |C               |2.000000                      |2.000000                   |
|10675<>XTREME 160<>DRUM<>SELF<>CA

In [91]:
#Series in prediction output but not in historical sales

missing_series = pred_itr_3_sf.join(
    final_output,
    final_output["PARENT_DEALER_CODE_MODEL_FAMILY"] == pred_itr_3_sf["PARENT_DEALER_CODE_MODEL_FAMILY"],
    "leftanti"
)

missing_series.select(F.count_distinct("PARENT_DEALER_CODE_MODEL_FAMILY").alias("SERIES_IN_PREDICTIONS_BUT_NOT_IN_HISTORICAL_OUTPUT")).show()

------------------------------------------------------
|"SERIES_IN_PREDICTIONS_BUT_NOT_IN_HISTORICAL_OU...  |
------------------------------------------------------
|31185                                               |
------------------------------------------------------



In [92]:
missing_series.select(F.sum("PRED_MEAN").alias("TOTAL_PREDICTED_SALES")).show()

---------------------------
|"TOTAL_PREDICTED_SALES"  |
---------------------------
|258895.0932658349        |
---------------------------



In [93]:
pred_itr_3_sf.select(F.count_distinct("PARENT_DEALER_CODE_MODEL_FAMILY")).show()

------------------------------------------------------
|"COUNT( DISTINCT ""PARENT_DEALER_CODE_MODEL_FAM...  |
------------------------------------------------------
|31893                                               |
------------------------------------------------------



In [95]:
missing_series = pred_itr_3_sf.join(
    actual_sales_df,
    actual_sales_df["PARENT_DEALER_CODE_MODEL_FAMILY"] == pred_itr_3_sf["PARENT_DEALER_CODE_MODEL_FAMILY"],
    "leftanti"
)

missing_series.select(F.count_distinct("PARENT_DEALER_CODE_MODEL_FAMILY").alias("SERIES_IN_PREDICTIONS_BUT_NOT_IN_ACTUAL_SALES")).show()

---------------------------------------------------
|"SERIES_IN_PREDICTIONS_BUT_NOT_IN_ACTUAL_SALES"  |
---------------------------------------------------
|31479                                            |
---------------------------------------------------



In [123]:
from snowflake.snowpark import Window
from snowflake.snowpark import functions as F

# ============================================================
# 1. SKU Supercedence + Model Family Mapping
# ============================================================
def fetchSKUSupercedence_snowpark(session):
    data = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")
    data_1 = session.table("MOP_DATABASE.SOQ.MODEL_FAMILY_MAPPING")
    result = data.join(data_1, on="MODEL", how="left")
    result = result.with_column("SKU_UNIQUE_FAMILY_CODE", F.col("UNIQUEFAMILYCODE"))
    for old_col in result.columns:
        new_col = old_col.replace('"', '')
        result = result.rename(old_col, new_col)
    result = result.with_column(
        "MODEL_FAMILY_CODE",
        F.concat(
            F.col("MODEL_FAMILY"),
            F.lit('<>'),
            F.substring(
                F.col("UNIQUEFAMILYCODE"),
                F.charindex(F.lit('<>'), F.col("UNIQUEFAMILYCODE")) + F.lit(2)
            )
        )
    )
    result = result.rename("UNIQUEFAMILYCODE", "UNIQUE FAMILY CODE")
    return result

result = fetchSKUSupercedence_snowpark(session)

# ============================================================
# 2. Models for Forecasting
# ============================================================
def return_models_for_forecasting(session, use_selected_models):
    if not use_selected_models:
        return None
    models_for_forecasting = session.table('MOP_DATABASE.SOQ.MODELS_FOR_FORECASTING').to_pandas()
    return models_for_forecasting["MODEL_NAME"].tolist()

name_of_models = return_models_for_forecasting(session, True)

# ============================================================
# 3. ECR Sales
# ============================================================
def get_ecr_sales_snowpark(session, customer_types, start_date, end_date, name_of_models):
    ecr_sales = session.table("ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS") \
        .filter(F.col("X_CUSTOMER_TYPE").in_(customer_types)) \
        .filter((F.col("CAL_DATE") >= F.lit(start_date)) & (F.col("CAL_DATE") <= F.lit(end_date)))
    if name_of_models is not None:
        ecr_sales = ecr_sales.filter(F.col("MODEL").isin(name_of_models))
    ecr_sales = ecr_sales.with_column(
        "NET_SALES",
        F.when(
            (F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")) < 0,
            F.lit(0)
        ).otherwise(
            F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")
        )
    )
    return ecr_sales

ecr_sales = get_ecr_sales_snowpark(
    session,
    start_date='2026-06-01',
    end_date='2026-08-31',
    customer_types=['Individual'],
    name_of_models=name_of_models
)
ecr_sales = ecr_sales.select('DEALER_CODE', 'CAL_DATE', 'SKU', 'NET_SALES')

# ============================================================
# 4. Active SKU Supercedence
# ============================================================
active_sku_supercedence = result.filter(F.lower(F.col("SKUSTATUS")) == 'active')
active_sku_supercedence = active_sku_supercedence.select("SKU", "MODEL_FAMILY_CODE")

# ============================================================
# 5. OBD Mapping - Map old SKUs to current active SKUs
# ============================================================
obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW")
sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")

obd_data_joined = obd_data.join(
    sku_supercedence.select("SKU", "SKUSTATUS"),
    obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"],
    how='left'
)
obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
    .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU")

ecr_sales = ecr_sales.join(
    obd_data_active_skus,
    ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"],
    how="left"
)
ecr_sales = ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))

# ============================================================
# 6. Aggregate to collapse multiple transaction rows
# ============================================================
ecr_sales = ecr_sales.group_by("DEALER_CODE", "CAL_DATE", "SKU") \
    .agg(F.sum("NET_SALES").alias("NET_SALES"))

# ============================================================
# 7. Join sales with active SKU supercedence (inner join)
# ============================================================
joined_df = ecr_sales.join(right=active_sku_supercedence, on=["SKU"])
joined_df = joined_df.select("DEALER_CODE", "CAL_DATE", "MODEL_FAMILY_CODE", "SKU", "NET_SALES")

# ============================================================
# 8. Parent Dealer Code from ORG_HIERARCHY (same as training procedure)
# ============================================================
parent_map = session.table("FIVETRAN_DATABASE.ORACLE_LDP_OLAP_SCHEMA.WC_INT_ORG_DH").select(
    F.col("X_DEALER_CODE_HIER").alias("DEALER_CODE"),
    F.trim(F.split_part(F.col("PAR_ORG_NAME"), F.lit("-"), F.lit(1))).alias("PARENT_DEALER_CODE")
).distinct()

# ============================================================
# 9. Family-level daily sales (dealer grain)
# ============================================================
family_level_daily_sales = parent_map.join(joined_df, on=["DEALER_CODE"])
family_level_daily_sales = family_level_daily_sales.with_column(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    F.concat(F.trim(F.col("PARENT_DEALER_CODE")), F.lit('<>'), F.col("MODEL_FAMILY_CODE"))
)

# ============================================================
# 10. Aggregate at parent_dealer+family and parent_dealer+family+SKU
# ============================================================
family_sales_3_months = family_level_daily_sales \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .agg(F.sum("NET_SALES").alias("LAST_3_MONTHS_FAMILY_SALES"))

sku_level_sales_3_months = family_level_daily_sales \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY", "SKU") \
    .agg(F.sum("NET_SALES").alias("LAST_3_MONTHS_SKU_SALES"))

family_and_sku = family_sales_3_months.join(
    right=sku_level_sales_3_months,
    on=["PARENT_DEALER_CODE_MODEL_FAMILY"]
)

# ============================================================
# 11. SKU Proportion (using DIV0 to avoid division by zero)
# ============================================================
sku_count = family_and_sku \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .agg(F.count("SKU").alias("SKU_COUNT"))

family_and_sku = family_and_sku.join(sku_count, on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

family_and_sku = family_and_sku.with_column(
    "PROPORTION",
    F.when(F.col("LAST_3_MONTHS_FAMILY_SALES") > 0,
           F.col("LAST_3_MONTHS_SKU_SALES") / F.col("LAST_3_MONTHS_FAMILY_SALES"))
    .when(F.col("SKU_COUNT") > 0,
           F.lit(1) / F.col("SKU_COUNT"))
    .otherwise(F.lit(0))
)

# ============================================================
# 12. ABC Classification (using DIV0 for cumulative pct)
# ============================================================
w = Window.partition_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .order_by(F.col("LAST_3_MONTHS_SKU_SALES").desc())
family_total = Window.partition_by("PARENT_DEALER_CODE_MODEL_FAMILY")

family_and_sku = family_and_sku.with_column(
    "CUMULATIVE_PCT",
    F.call_builtin(
        "DIV0",
        F.sum("LAST_3_MONTHS_SKU_SALES").over(
            w.rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW)
        ),
        F.sum("LAST_3_MONTHS_SKU_SALES").over(family_total)
    )
)

family_and_sku = family_and_sku.with_column(
    "ABC_CATEGORY",
    F.when(F.col("CUMULATIVE_PCT") <= 0.80, F.lit("A"))
     .when(F.col("CUMULATIVE_PCT") <= 0.95, F.lit("B"))
     .otherwise(F.lit("C"))
)

# ============================================================
# 13. Final output - MATERIALIZE to break the lazy DAG
# ============================================================
final_output = family_and_sku.select(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    "SKU",
    "PROPORTION",
    "ABC_CATEGORY",
    "LAST_3_MONTHS_FAMILY_SALES",
    "LAST_3_MONTHS_SKU_SALES"
)
final_output.write.mode("overwrite").save_as_table("MOP_DATABASE.SOQ.TEMP_FINAL_OUTPUT")
final_output = session.table("MOP_DATABASE.SOQ.TEMP_FINAL_OUTPUT")

# ============================================================
# 14. Actual Sales (Sep 1-22) - same parent dealer source
# ============================================================
actual_ecr_sales = get_ecr_sales_snowpark(
    session,
    start_date='2026-09-01',
    end_date='2026-09-22',
    customer_types=['Individual'],
    name_of_models=name_of_models
)
actual_ecr_sales = actual_ecr_sales.select('DEALER_CODE', 'CAL_DATE', 'SKU', 'NET_SALES')

actual_ecr_sales = actual_ecr_sales.join(
    obd_data_active_skus,
    actual_ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"],
    how="left"
)
actual_ecr_sales = actual_ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))

actual_ecr_sales = actual_ecr_sales.group_by("DEALER_CODE", "CAL_DATE", "SKU") \
    .agg(F.sum("NET_SALES").alias("NET_SALES"))

actual_joined_df = actual_ecr_sales.join(right=active_sku_supercedence, on=["SKU"])
actual_joined_df = actual_joined_df.select("DEALER_CODE", "CAL_DATE", "MODEL_FAMILY_CODE", "SKU", "NET_SALES")

actual_joined_df_with_pdc = parent_map.join(actual_joined_df, on=["DEALER_CODE"])
actual_joined_df_with_pdc = actual_joined_df_with_pdc.with_column(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    F.concat(F.trim(F.col("PARENT_DEALER_CODE")), F.lit('<>'), F.col("MODEL_FAMILY_CODE"))
)

actual_sales_df = actual_joined_df_with_pdc.select(
    "PARENT_DEALER_CODE_MODEL_FAMILY", "SKU", "CAL_DATE", "NET_SALES"
)

# ============================================================
# 15. Join predictions with proportions for SKU disaggregation
# ============================================================
pred_itr_3 = pd.read_parquet(r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#3_feature_engineering\Modelling\predictions_2026_negbin_torch_convention\20260922_153432_34f1cdf5\DA7CA8~1.PAR")
pred_itr_3_mean = pred_itr_3[["PARENT_DEALER_CODE_MODEL_FAMILY", "CAL_DATE", "PRED_MEAN"]]
pred_itr_3_sf = session.create_dataframe(pred_itr_3_mean)

pred_itr_3_sf = pred_itr_3_sf.with_column(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    F.replace(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"), F.lit('_'), F.lit('<>'))
)

join_with_predictions = final_output.join(pred_itr_3_sf, on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

join_with_predictions_sf = join_with_predictions.select(
    'PARENT_DEALER_CODE_MODEL_FAMILY', 'SKU', 'CAL_DATE',
    'PRED_MEAN', 'PROPORTION', 'ABC_CATEGORY'
)

join_with_predictions_sf = join_with_predictions_sf.with_column(
    "SKU_PREDICTION", F.col("PROPORTION") * F.col("PRED_MEAN")
)

# ============================================================
# 16. Verification
# ============================================================
join_with_predictions_sf.select(
    F.sum("SKU_PREDICTION").alias("TOTAL_SKU_PREDICTION")
).show()

pred_itr_3_sf.select(
    F.sum("PRED_MEAN").alias("TOTAL_FAMILY_PREDICTION")
).show()

# ============================================================
# 17. Aggregate actual sales at PARENT_DEALER_CODE_MODEL_FAMILY + SKU + CAL_DATE
# ============================================================
actual_sales_agg = actual_sales_df.group_by("PARENT_DEALER_CODE_MODEL_FAMILY", "SKU", "CAL_DATE") \
    .agg(F.sum("NET_SALES").alias("ACTUAL_NET_SALES"))

# ============================================================
# 18. Join actuals with SKU-level predictions
# ============================================================
accuracy_df = join_with_predictions_sf.join(
    actual_sales_agg,
    on=["PARENT_DEALER_CODE_MODEL_FAMILY", "SKU", "CAL_DATE"],
    how="left"
)
accuracy_df = accuracy_df.with_column("ACTUAL_NET_SALES", F.coalesce(F.col("ACTUAL_NET_SALES"), F.lit(0)))

# Materialize to break lazy DAG before accuracy calculations
accuracy_df.write.mode("overwrite").save_as_table("MOP_DATABASE.SOQ.TEMP_ACCURACY_DF")

# ============================================================
# 19. Accuracy by ABC Category (pure SQL to avoid Snowpark division issues)
# ============================================================
session.sql("""
    SELECT 
        ABC_CATEGORY,
        SUM(SKU_PREDICTION) AS TOTAL_PREDICTED,
        SUM(ACTUAL_NET_SALES) AS TOTAL_ACTUAL,
        COUNT(DISTINCT PARENT_DEALER_CODE_MODEL_FAMILY || '|' || SKU) AS DISTINCT_FAMILY_SKU_COMBOS,
        SUM(ABS(ACTUAL_NET_SALES - SKU_PREDICTION)) AS TOTAL_ABS_ERROR,
        ROUND(DIV0(SUM(ABS(ACTUAL_NET_SALES - SKU_PREDICTION)), SUM(ACTUAL_NET_SALES)) * 100, 2) AS MAPE_PROXY,
        ROUND(DIV0(SUM(SKU_PREDICTION) - SUM(ACTUAL_NET_SALES), SUM(ACTUAL_NET_SALES)) * 100, 2) AS BIAS_PCT
    FROM MOP_DATABASE.SOQ.TEMP_ACCURACY_DF
    GROUP BY ABC_CATEGORY
    ORDER BY ABC_CATEGORY
""").show()

--------------------------
|"TOTAL_SKU_PREDICTION"  |
--------------------------
|2049736.491768682       |
--------------------------

-----------------------------
|"TOTAL_FAMILY_PREDICTION"  |
-----------------------------
|2073623.4918465866         |
-----------------------------

----------------------------------------------------------------------------------------------------------------------------------------
|"ABC_CATEGORY"  |"TOTAL_PREDICTED"   |"TOTAL_ACTUAL"  |"DISTINCT_FAMILY_SKU_COMBOS"  |"TOTAL_ABS_ERROR"   |"MAPE_PROXY"  |"BIAS_PCT"  |
----------------------------------------------------------------------------------------------------------------------------------------
|A               |419076.5829326557   |56175.000000    |6199                          |424971.42628362915  |756.51        |646.02      |
|C               |1403850.1202953474  |189849.000000   |24662                         |1405100.9854534152  |740.12        |639.46      |
|B               |226809.788

In [124]:
join_with_predictions_sf.show()

-----------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"        |"SKU"           |"CAL_DATE"           |"PRED_MEAN"          |"PROPORTION"    |"ABC_CATEGORY"  |"SKU_PREDICTION"      |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------
|10479<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK  |HGLATDRSCFITBK  |2026-09-20 00:00:00  |0.0900355128653263   |0.571428571429  |A               |0.05144886449451076   |
|10479<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK  |HGLATDRSCFIBMS  |2026-09-20 00:00:00  |0.0900355128653263   |0.428571428571  |C               |0.03858664837081554   |
|10479<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK  |HGLATDRSCFITBK  |2026-09-21 00:00:00  |0.16413754340795314  |0.571428571429  |A               |0.09379288194747215   |
|10479<>GLAMOUR<>DRUM<>SELF<